In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
%cd /content

!rm -rf /content/smallnet
!git clone https://github.com/SepehrAkbari/smallnet.git

%cd /content/smallnet

!git status --short
!git log -1 --oneline

/content
Cloning into 'smallnet'...
remote: Enumerating objects: 2150, done.
remote: Counting objects: 100% (265/265), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 2150 (delta 149), reused 171 (delta 64), pack-reused 1885 (from 1)
Receiving objects: 100% (2150/2150), 582.51 MiB | 27.31 MiB/s, done.
Resolving deltas: 100% (384/384), done.
Updating files: 100% (1664/1664), done.
/content/smallnet
869ade6 (HEAD -> main, origin/main, origin/HEAD) restructure + val


In [19]:
!wget -qO- https://astral.sh/uv/install.sh | sh

!uv --version

downloading uv 0.11.31 x86_64-unknown-linux-gnu
installing to /usr/local/bin
  uv
  uvx
everything's installed!
uv 0.11.31 (x86_64-unknown-linux-gnu)


In [20]:
!uv --version
!uv sync

uv 0.11.31 (x86_64-unknown-linux-gnu)
Using CPython 3.12.13 interpreter at: /usr/bin/python3
Creating virtual environment at: .venv
Resolved 85 packages in 1ms
Prepared 79 packages in 46.30s                                           
Installed 79 packages in 250ms                              
 + asttokens==3.0.1
 + comm==0.2.3
 + contourpy==1.3.3
 + cuda-bindings==13.2.0
 + cuda-pathfinder==1.5.4
 + cuda-toolkit==13.0.2
 + cycler==0.12.1
 + debugpy==1.8.20
 + decorator==5.3.1
 + executing==2.2.1
 + filelock==3.29.0
 + fonttools==4.63.0
 + fsspec==2026.4.0
 + iniconfig==2.3.0
 + ipykernel==7.2.0
 + ipython==9.13.0
 + ipython-pygments-lexers==1.1.1
 + ipywidgets==8.1.8
 + jedi==0.20.0
 + jinja2==3.1.6
 + jupyter-client==8.8.0
 + jupyter-core==5.9.1
 + jupyterlab-widgets==3.0.16
 + kiwisolver==1.5.0
 + markupsafe==3.0.3
 + matplotlib==3.10.9
 + matplotlib-inline==0.2.2
 + mpmath==1.3.0
 + nest-asyncio==1.6.0
 + networkx==3.6.1
 + numpy==2.4.4
 + nvidia-cublas==13.1.1.3
 + nvidia-cuda-cup

In [5]:
from pathlib import Path
import subprocess

repo = Path("/content/smallnet")
backup = Path(
    "/content/drive/MyDrive/smallnet_colab_backup"
)

assert backup.exists(), f"Backup directory not found: {backup}"

restore_pairs = [
    (
        backup / "camvid_vgg_cp",
        repo / "results/camvid_vgg_cp",
    ),
    (
        backup / "paper",
        repo / "results/paper",
    ),
]

for source, destination in restore_pairs:
    if source.exists():
        destination.mkdir(parents=True, exist_ok=True)

        subprocess.run(
            [
                "rsync",
                "-a",
                f"{source}/",
                f"{destination}/",
            ],
            check=True,
        )

        print(f"Restored: {source}")
    else:
        print(f"Not found, skipped: {source}")

# Prevent the old accidental duplicate nesting from returning.
duplicate = (
    repo
    / "results/camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate.exists():
    subprocess.run(
        ["rm", "-rf", str(duplicate)],
        check=True,
    )
    print("Removed accidental duplicate nesting.")

Restored: /content/drive/MyDrive/smallnet_colab_backup/camvid_vgg_cp
Restored: /content/drive/MyDrive/smallnet_colab_backup/paper


In [7]:
from pathlib import Path
import shutil

duplicate_backup = Path(
    "/content/drive/MyDrive/"
    "smallnet_colab_backup/"
    "camvid_vgg_cp/camvid_vgg_cp"
)

if duplicate_backup.exists():
    shutil.rmtree(duplicate_backup)
    print("Removed duplicate nesting from Drive backup.")
else:
    print("No duplicate Drive directory found.")

No duplicate Drive directory found.


In [9]:
from pathlib import Path
import shutil
import subprocess

drive_root = Path("/content/drive/MyDrive")
repo = Path("/content/smallnet")

model_candidates = [
    drive_root
    / "smallnet_colab_backup/model/best_model.pth",
    drive_root
    / "smallnet/model/best_model.pth",
]

model_source = next(
    (
        path
        for path in model_candidates
        if path.is_file()
    ),
    None,
)

if model_source is None:
    model_matches = list(
        drive_root.rglob("best_model.pth")
    )
    model_source = (
        model_matches[0]
        if model_matches
        else None
    )

assert model_source is not None, (
    "Could not find best_model.pth in Google Drive."
)

model_destination = (
    repo / "model/best_model.pth"
)
model_destination.parent.mkdir(
    parents=True,
    exist_ok=True,
)
shutil.copy2(
    model_source,
    model_destination,
)

print("Model source:", model_source)
print("Model copied to:", model_destination)

Model source: /content/drive/MyDrive/model/best_model.pth
Model copied to: /content/smallnet/model/best_model.pth


In [10]:
from pathlib import Path

repo = Path("/content/smallnet")

required = [
    repo / "model/best_model.pth",
    repo / "data/CamVid/class_dict.csv",
    repo / "data/CamVid/train",
    repo / "data/CamVid/train_labels",
    repo / "data/CamVid/val",
    repo / "data/CamVid/val_labels",
    repo / "data/CamVid/test",
    repo / "data/CamVid/test_labels",
    repo
    / "results/camvid_vgg_cp/"
    "dataset_validation_report.json",
]

for path in required:
    print(
        "OK" if path.exists() else "MISSING",
        path,
    )

assert all(path.exists() for path in required)

OK /content/smallnet/model/best_model.pth
OK /content/smallnet/data/CamVid/class_dict.csv
OK /content/smallnet/data/CamVid/train
OK /content/smallnet/data/CamVid/train_labels
OK /content/smallnet/data/CamVid/val
OK /content/smallnet/data/CamVid/val_labels
OK /content/smallnet/data/CamVid/test
OK /content/smallnet/data/CamVid/test_labels
OK /content/smallnet/results/camvid_vgg_cp/dataset_validation_report.json


In [11]:
from pathlib import Path

camvid = Path("/content/smallnet/data/CamVid")

for split in ["train", "val", "test"]:
    image_count = len(
        list((camvid / split).glob("*"))
    )
    mask_count = len(
        list(
            (
                camvid / f"{split}_labels"
            ).glob("*")
        )
    )

    print(
        split,
        "images:",
        image_count,
        "masks:",
        mask_count,
    )

train images: 369 masks: 369
val images: 100 masks: 100
test images: 232 masks: 232


In [12]:
!nvidia-smi

import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0),
    )

assert torch.cuda.is_available()

Thu Jul 23 20:16:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [13]:
import json
from pathlib import Path

config_path = Path(
    "/content/smallnet/"
    "configs/camvid_vgg_cp_paper.json"
)

config = json.loads(
    config_path.read_text()
)

print(
    json.dumps(
        config.get("final_structural", {}),
        indent=2,
    )
)

{
  "ranks": [
    32,
    64,
    128,
    256,
    512
  ],
  "seeds": [
    0,
    1,
    2
  ],
  "iteration_budget": 200,
  "init": "random",
  "memory_efficient_mttkrp": true,
  "mttkrp_rank_chunk_size": 64,
  "mttkrp_max_explicit_bytes": 536870912,
  "numerical_tolerance": 1e-05,
  "residual_output_chunk_size": 8,
  "factor_diagnostic_thresholds": {
    "near_zero_component_norm_threshold": 1e-12,
    "extreme_factor_norm_threshold": 1000000.0,
    "scaling_spread_threshold": 1000000.0,
    "cancellation_ratio_threshold": 1000.0,
    "bounded_reconstruction_multiple": 10.0
  },
  "output_dir": "results/camvid_vgg_cp/final_structural",
  "figures_dir": "results/paper/figures",
  "audit_path": "results/paper/final_structural_audit.md"
}


In [21]:
%cd /content/smallnet

!uv run python -m pytest -q

/content/smallnet
..............................................s......................... [ 81%]
................                                                         [100%]
=============================== warnings summary ===============================
.venv/lib/python3.12/site-packages/tensorly/solvers/nnls.py:103
  /content/smallnet/.venv/lib/python3.12/site-packages/tensorly/solvers/nnls.py:103: SyntaxWarning: invalid escape sequence '\l'
    .. math:: \lambda_s, \lambda_r

.venv/lib/python3.12/site-packages/tensorly/solvers/admm.py:99
  /content/smallnet/.venv/lib/python3.12/site-packages/tensorly/solvers/admm.py:99: SyntaxWarning: invalid escape sequence '\_'
    .. math:: dual\_var = dual\_var + (Ax + Bx_{split} - c)

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
87 passed, 1 skipped, 2 warnings in 48.88s


In [22]:
from pathlib import Path
import subprocess

REPO = Path("/content/smallnet")
BACKUP = Path(
    "/content/drive/MyDrive/"
    "smallnet_colab_backup"
)

def backup_results():
    pairs = [
        (
            REPO / "results/camvid_vgg_cp",
            BACKUP / "camvid_vgg_cp",
        ),
        (
            REPO / "results/paper",
            BACKUP / "paper",
        ),
    ]

    for source, destination in pairs:
        destination.mkdir(
            parents=True,
            exist_ok=True,
        )

        subprocess.run(
            [
                "rsync",
                "-a",
                f"{source}/",
                f"{destination}/",
            ],
            check=True,
        )

    duplicate = (
        BACKUP
        / "camvid_vgg_cp/camvid_vgg_cp"
    )

    if duplicate.exists():
        subprocess.run(
            ["rm", "-rf", str(duplicate)],
            check=True,
        )

    print("Results backed up to Google Drive.")

In [23]:
backup_results()

Results backed up to Google Drive.


In [24]:
import csv
import subprocess
from pathlib import Path

def show_progress():
    summary_path = (
        REPO
        / "results/camvid_vgg_cp/"
        "final_structural/"
        "final_structural_summary.csv"
    )

    if not summary_path.exists():
        print("No summary written yet.")
        return

    with summary_path.open(newline="") as file:
        rows = list(csv.DictReader(file))

    completed = [
        row
        for row in rows
        if row.get("status") == "completed"
    ]

    cp_rows = [
        row
        for row in completed
        if row.get("method")
        == "cp_200_iterations"
    ]

    svd_rows = [
        row
        for row in completed
        if row.get("method")
        == "matrix_svd_output_unfolding"
    ]

    print("Total rows:", len(rows))
    print("Completed rows:", len(completed))
    print("Completed CP rows:", len(cp_rows))
    print("Completed SVD rows:", len(svd_rows))

    completed_by_rank = {}

    for row in completed:
        rank = row.get("rank")
        completed_by_rank.setdefault(
            rank,
            [],
        ).append(
            (
                row.get("method"),
                row.get("seed"),
            )
        )

    for rank in sorted(
        completed_by_rank,
        key=int,
    ):
        print(
            f"Rank {rank}:",
            completed_by_rank[rank],
        )


def run_final_structural_rank(rank):
    command = [
        "uv",
        "run",
        "python",
        "scripts/run_experiment.py",
        "--config",
        "configs/camvid_vgg_cp_paper.json",
        "--stage",
        "final-structural",
        "--device",
        "cuda",
        "--ranks",
        str(rank),
        "--seeds",
        "0",
        "1",
        "2",
    ]

    print(
        "Running:",
        " ".join(command),
    )

    try:
        subprocess.run(
            command,
            cwd=REPO,
            check=True,
        )
    finally:
        backup_results()
        show_progress()

In [25]:
run_final_structural_rank(32)

Running: uv run python scripts/run_experiment.py --config configs/camvid_vgg_cp_paper.json --stage final-structural --device cuda --ranks 32 --seeds 0 1 2
Results backed up to Google Drive.
Total rows: 4
Completed rows: 3
Completed CP rows: 3
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]


In [26]:
import pandas as pd
import json
from pathlib import Path

root = Path(
    "/content/smallnet/results/"
    "camvid_vgg_cp/final_structural"
)

summary = pd.read_csv(
    root / "final_structural_summary.csv"
)

failed = summary[
    summary["status"] != "completed"
].copy()

display(
    failed[
        [
            "method",
            "rank",
            "seed",
            "status",
            "failure_exception",
        ]
    ]
)

metadata = json.loads(
    (
        root
        / "final_structural_metadata.json"
    ).read_text()
)

print("\nCurrently failed scientific rows:")
print(
    json.dumps(
        metadata.get(
            "currently_failed_scientific_rows",
            [],
        ),
        indent=2,
    )
)

print("\nFailure history:")
print(
    json.dumps(
        metadata.get("failure_history", []),
        indent=2,
    )
)

,method,rank,seed,status,failure_exception
0,matrix_svd_output_unfolding,32,NaN,failed,AssertionError('Matrix-SVD residual does not m...



Currently failed scientific rows:
[
  {
    "method": "matrix_svd_output_unfolding",
    "rank": 32,
    "seed": "",
    "iteration_budget": "",
    "status": "failed",
    "failure_exception": "AssertionError('Matrix-SVD residual does not match output tail')",
    "checkpoint_sha256": "1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3",
    "dataset_validation_report_sha256": "f1bb932fa776a2065507de6b29bdc174c139a1e20f8c4172e04df74af037f57b",
    "target_tensor_sha256": "fb442eda032751bbfd8209ab34c88301f4fe2d320b2c43ef56e911094c2f94ef"
  }
]

Failure history:
[
  {
    "method": "matrix_svd_output_unfolding",
    "rank": 32,
    "seed": "",
    "iteration_budget": "",
    "status": "failed",
    "failure_exception": "AssertionError('Matrix-SVD residual does not match output tail')",
    "checkpoint_sha256": "1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3",
    "dataset_validation_report_sha256": "f1bb932fa776a2065507de6b29bdc174c139a1e20f8c4172e04df74

In [27]:
if len(failed) == 1:
    for column, value in failed.iloc[0].items():
        print(f"{column}: {value}")

activation_absolute_mse: nan
activation_cosine_similarity: nan
activation_element_count: nan
activation_example_count: nan
activation_example_normalized_squared_error_max: nan
activation_example_normalized_squared_error_mean: nan
activation_example_normalized_squared_error_min: nan
activation_example_normalized_squared_error_std_population: nan
activation_input_protocol: nan
activation_normalized_squared_error: nan
activation_relative_frobenius_error: nan
activation_scope: nan
actual_relative_frobenius_error: nan
actual_relative_squared_frobenius_error: nan
bounded_reconstruction_multiple: nan
cancellation_ratio_threshold: nan
checkpoint_sha256: 1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3
component_contribution_max_to_reconstructed_norm_ratio: nan
component_contribution_norm_max: nan
component_contribution_norm_min: nan
component_contribution_norms_json: nan
component_contribution_sum_to_reconstructed_norm_ratio: nan
component_log_norm_product_max: nan
component_lo

In [28]:
import torch

checkpoint = torch.load(
    "/content/smallnet/model/best_model.pth",
    map_location="cpu",
)

state_dict = (
    checkpoint["state_dict"]
    if isinstance(checkpoint, dict)
    and "state_dict" in checkpoint
    else checkpoint
)

candidate_keys = [
    key
    for key in state_dict
    if key.endswith("classifier.0.weight")
]

print("Candidate keys:", candidate_keys)

assert len(candidate_keys) == 1

weight = state_dict[
    candidate_keys[0]
].detach()

matrix = weight.reshape(
    weight.shape[0],
    -1,
)

print("Weight shape:", tuple(weight.shape))
print("Matrix shape:", tuple(matrix.shape))
print("CPU dtype:", matrix.dtype)

# CPU SVD sanity check
u, s, vh = torch.linalg.svd(
    matrix.cpu(),
    full_matrices=False,
)

print(
    "CPU SVD passed:",
    tuple(u.shape),
    tuple(s.shape),
    tuple(vh.shape),
)

Candidate keys: ['classifier.0.weight']
Weight shape: (4096, 512, 7, 7)
Matrix shape: (4096, 25088)
CPU dtype: torch.float32
CPU SVD passed: (4096, 4096) (4096,) (4096, 25088)


In [29]:
%cd /content/smallnet

!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage final-structural \
  --device cuda \
  --ranks 32 \
  --seeds 0 1 2

/content/smallnet
Wrote:
  /content/smallnet/results/camvid_vgg_cp/final_structural/final_structural_metadata.json


In [30]:
show_progress()

Total rows: 4
Completed rows: 3
Completed CP rows: 3
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]


In [37]:
%%bash
set -e

cd /content

if [ -d smallnet/.git ]; then
    cd smallnet
    git fetch origin
    git pull --ff-only origin main
else
    print "smallnet/.git not found, skipping git pull."
fi

Updating f6a44a0..e2d9816
Fast-forward
 colab/n5.ipynb                | 1083 ++++++++++++++++++++++++++++++++++++++++-
 src/smallnet/factorization.py |    2 +-
 2 files changed, 1070 insertions(+), 15 deletions(-)


From https://github.com/SepehrAkbari/smallnet
   f6a44a0..e2d9816  main       -> origin/main
From https://github.com/SepehrAkbari/smallnet
 * branch            main       -> FETCH_HEAD


In [40]:
!pip install -U tensorly-torch
!pip install tensorly

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 18.3 MB/s eta 0:00:00a 0:00:01


In [41]:
%cd /content/smallnet

import inspect
import math
import pandas as pd
import torch
from pathlib import Path


import src.smallnet.experiment as experiment_module

# --------------------------------------------------
# 1. Locate the exact assertion in the source
# --------------------------------------------------

source_path = Path(
    inspect.getsourcefile(experiment_module)
)

lines = source_path.read_text().splitlines()

needle = "Matrix-SVD residual does not match output tail"

matches = [
    index
    for index, line in enumerate(lines)
    if needle in line
]

print("Source:", source_path)
print("Assertion line indexes:", matches)

for index in matches:
    start = max(0, index - 25)
    end = min(len(lines), index + 15)

    print("\n" + "=" * 80)

    for line_number in range(start, end):
        print(
            f"{line_number + 1:5d}: "
            f"{lines[line_number]}"
        )

# --------------------------------------------------
# 2. Load the target weight
# --------------------------------------------------

checkpoint = torch.load(
    "model/best_model.pth",
    map_location="cpu",
)

state_dict = (
    checkpoint["state_dict"]
    if isinstance(checkpoint, dict)
    and "state_dict" in checkpoint
    else checkpoint
)

weight_key = next(
    key
    for key in state_dict
    if key.endswith("classifier.0.weight")
)

weight = (
    state_dict[weight_key]
    .detach()
    .cpu()
    .to(torch.float64)
)

matrix = weight.reshape(
    weight.shape[0],
    -1,
)

rank = 32

print("\nWeight key:", weight_key)
print("Matrix shape:", tuple(matrix.shape))
print("Diagnostic dtype:", matrix.dtype)

# --------------------------------------------------
# 3. Compute a high-precision rank-32 SVD
# --------------------------------------------------

u, s, vh = torch.linalg.svd(
    matrix,
    full_matrices=False,
)

total_energy = torch.sum(s.square())
tail_energy = torch.sum(s[rank:].square())

spectral_tail = (
    tail_energy / total_energy
).item()

approximation = (
    u[:, :rank]
    * s[:rank].unsqueeze(0)
) @ vh[:rank, :]

direct_residual = (
    torch.sum(
        (matrix - approximation).square()
    )
    / torch.sum(matrix.square())
).item()

difference = abs(
    direct_residual - spectral_tail
)

print("\nIndependent float64 check")
print("Spectral tail:    ", repr(spectral_tail))
print("Direct residual:  ", repr(direct_residual))
print("Absolute diff:    ", repr(difference))
print("Diff <= 1e-5:     ", difference <= 1e-5)
print("Diff <= 1e-6:     ", difference <= 1e-6)
print("Diff <= 1e-7:     ", difference <= 1e-7)

# --------------------------------------------------
# 4. Show the failed stored row in full
# --------------------------------------------------

summary_path = Path(
    "results/camvid_vgg_cp/"
    "final_structural/"
    "final_structural_summary.csv"
)

summary = pd.read_csv(summary_path)

failed_svd = summary[
    (summary["method"] == "matrix_svd_output_unfolding")
    & (pd.to_numeric(
        summary["rank"],
        errors="coerce",
    ) == 32)
]

print("\nStored rank-32 SVD row:")

for _, row in failed_svd.iterrows():
    for column, value in row.items():
        if (
            "residual" in column.lower()
            or "tail" in column.lower()
            or "error" in column.lower()
            or column in {
                "method",
                "rank",
                "status",
                "failure_exception",
            }
        ):
            print(f"{column}: {value}")

/content/smallnet
Source: /content/smallnet/src/smallnet/experiment.py
Assertion line indexes: [2443]

 2419:                 previous
 2420:                 and previous.get("status") == "completed"
 2421:                 and accepted.get("status") != "completed"
 2422:             ):
 2423:                 continue
 2424:             persisted[key] = accepted
 2425:         write_csv(summary_path, [*persisted.values(), *preserved])
 2426: 
 2427:     for rank in ranks:
 2428:         key = (FINAL_SVD_METHOD, rank, "", "")
 2429:         if key in completed_keys:
 2430:             continue
 2431:         try:
 2432:             model = context["load_model"]()
 2433:             dense_conv = get_module(model, context["target_layer"])
 2434:             replacement = MatrixLowRankConv2d.from_svd(
 2435:                 dense_conv, rank, u, singular_values, vh
 2436:             )
 2437:             approximation = replacement.composed_kernel().detach().cpu()
 2438:             matrix_k

In [42]:
%cd /content/smallnet

!grep -R -n \
  -E "Matrix-SVD residual|output.*tail|output_mode_svd|svd_residual" \
  src/smallnet/final_structural.py \
  src/smallnet/experiment.py \
  src/smallnet/reconstruction.py \
  src/smallnet/structural.py \
  2>/dev/null

/content/smallnet
src/smallnet/experiment.py:51:    output_mode_svd,
src/smallnet/experiment.py:1739:            output_svd = float(stability_cfg.get("synthetic_output_mode_svd_residual_squared", 0.0))
src/smallnet/experiment.py:1747:            output_svd = float(reference["output_mode_svd_residual_squared"])
src/smallnet/experiment.py:1808:                    output_mode_svd_residual_squared=output_svd,
src/smallnet/experiment.py:2394:    u, singular_values, vh, svd_runtime, svd_device = output_mode_svd(
src/smallnet/experiment.py:2444:                raise AssertionError("Matrix-SVD residual does not match output tail")
src/smallnet/experiment.py:2470:                "initializer": "deterministic_output_mode_svd",
src/smallnet/experiment.py:2481:                "output_mode_tail_bound_squared": output_bound,
src/smallnet/experiment.py:2714:                    "output_mode_tail_bound_squared": output_bound,
src/smallnet/experiment.py:3016:        "output_mode_tail_bound_squared": 0.0

In [43]:
%cd /content/smallnet

!git pull --ff-only origin main
!uv sync

!uv run python -c "import tensorly; import tltorch; print('Dependencies: PASS')"
!uv run python -m pytest -q

/content/smallnet
remote: Enumerating objects: 25, done.
remote: Counting objects: 100% (25/25), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 15 (delta 11), reused 15 (delta 11), pack-reused 0 (from 0)
Unpacking objects: 100% (15/15), 11.33 KiB | 1.26 MiB/s, done.
From https://github.com/SepehrAkbari/smallnet
 * branch            main       -> FETCH_HEAD
   e2d9816..470d318  main       -> origin/main
Updating e2d9816..470d318
Fast-forward
 colab/n5.ipynb                                 | 190 ++++++++++++++++++++++---
 docs/final_structural_svd_tail_verification.md |  54 +++++++
 docs/reconstruction_experiment.md              |   6 +
 src/smallnet/experiment.py                     | 124 +++++++++++++++-
 src/smallnet/final_structural.py               | 113 +++++++++++++++
 tests/test_final_structural.py                 | 149 ++++++++++++++++++-
 6 files changed, 608 insertions(+), 28 deletions(-)
 create mode 100644 docs/final_structural_svd_tail_verification.md
Re

In [44]:
!uv run python scripts/run_experiment.py \
  --config configs/camvid_vgg_cp_paper.json \
  --stage final-structural \
  --device cuda \
  --ranks 32 \
  --seeds 0 1 2

Wrote:
  /content/smallnet/results/camvid_vgg_cp/final_structural/final_structural_metadata.json


In [45]:
show_progress()

Total rows: 4
Completed rows: 3
Completed CP rows: 3
Completed SVD rows: 0
Rank 32: [('cp_200_iterations', '0'), ('cp_200_iterations', '1'), ('cp_200_iterations', '2')]


In [46]:
import pandas as pd
import json
from pathlib import Path

root = Path(
    "/content/smallnet/results/"
    "camvid_vgg_cp/final_structural"
)

summary = pd.read_csv(
    root / "final_structural_summary.csv"
)

failed = summary[
    summary["status"] != "completed"
].copy()

display(
    failed[
        [
            "method",
            "rank",
            "seed",
            "status",
            "failure_exception",
        ]
    ]
)

metadata = json.loads(
    (
        root
        / "final_structural_metadata.json"
    ).read_text()
)

print("\nCurrently failed scientific rows:")
print(
    json.dumps(
        metadata.get(
            "currently_failed_scientific_rows",
            [],
        ),
        indent=2,
    )
)

print("\nFailure history:")
print(
    json.dumps(
        metadata.get("failure_history", []),
        indent=2,
    )
)

,method,rank,seed,status,failure_exception
3,matrix_svd_output_unfolding,32,NaN,failed,AssertionError('Matrix-SVD direct residual doe...



Currently failed scientific rows:
[
  {
    "method": "matrix_svd_output_unfolding",
    "rank": 32,
    "seed": "",
    "iteration_budget": "",
    "status": "failed",
    "failure_exception": "AssertionError('Matrix-SVD direct residual does not match the current same-SVD tail')",
    "checkpoint_sha256": "1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3",
    "dataset_validation_report_sha256": "f1bb932fa776a2065507de6b29bdc174c139a1e20f8c4172e04df74af037f57b",
    "target_tensor_sha256": "fb442eda032751bbfd8209ab34c88301f4fe2d320b2c43ef56e911094c2f94ef"
  }
]

Failure history:
[
  {
    "method": "matrix_svd_output_unfolding",
    "rank": 32,
    "seed": "",
    "iteration_budget": "",
    "status": "failed",
    "failure_exception": "AssertionError('Matrix-SVD residual does not match output tail')",
    "checkpoint_sha256": "1576b92ebb359a519d57d489f8ee823c033588fbd7c77b1c9091606f42fb74a3",
    "dataset_validation_report_sha256": "f1bb932fa776a2065507de6b29bdc174c1